In [1]:
import wrds
import pandas as pd

from pathlib import Path

In [2]:
conn = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


We extract monthly stock data from the CRSP Monthly Stock File (`msf_v2`) from January 1995 through December 2025. The query retrieves each security’s permanent identifier (`PERMNO`), month-end price, monthly return, shares outstanding, and market capitalization. We join the monthly stock file with CRSP’s security information history table to obtain identifying and classification variables and to restrict the sample to actively traded U.S. common equities listed on NYSE, AMEX, or NASDAQ. The resulting dataset contains the full eligible stock universe, rather than only the top 500 stocks, so that we can dynamically identify the 500 largest firms by market capitalization each month and retain their complete trailing 60-month return histories for portfolio construction.

In [3]:
START = "1995-01-01"
END_EXCLUSIVE = "2026-01-01"
N_STOCKS = 500
LOOKBACK = 60
FIRST_FORMATION = pd.Period("1999-12", freq="M")

In [4]:
query = f"""
SELECT m.permno, m.mthcaldt, m.mthprc, m.mthret, m.shrout, m.mthcap,
       EXISTS (
           SELECT 1
           FROM crsp.stksecurityinfohist AS s
           WHERE s.permno = m.permno
             AND m.mthcaldt BETWEEN s.secinfostartdt AND s.secinfoenddt
             AND s.sharetype = 'NS'
             AND s.securitytype = 'EQTY'
             AND s.securitysubtype = 'COM'
             AND s.usincflg = 'Y'
             AND s.issuertype IN ('ACOR', 'CORP')
             AND s.primaryexch IN ('N', 'A', 'Q')
       ) AS is_eligible
FROM crsp.msf_v2 AS m
WHERE m.mthcaldt >= '{START}'
  AND m.mthcaldt < '{END_EXCLUSIVE}'
"""

In [5]:
crsp = conn.raw_sql(query)

crsp.head()

,permno,mthcaldt,mthprc,mthret,shrout,mthcap,is_eligible
0,10001,1995-01-31,7.75,-0.03125,2224,17236.0,True
1,10002,1995-01-31,13.125,0.0,2999,39361.88,True
2,10003,1995-01-31,2.125,-0.055556,5038,10705.75,True
3,10009,1995-01-31,17.0,0.007407,1164,19788.0,True
4,10010,1995-01-31,5.25,0.05,10359,54384.75,True


In [6]:
crsp["month"] = pd.to_datetime(crsp["mthcaldt"]).dt.to_period("M")
for col in ["mthprc", "mthret", "shrout", "mthcap"]:
    crsp[col] = pd.to_numeric(crsp[col], errors="coerce")
crsp["is_eligible"] = crsp["is_eligible"].astype(bool)

In [7]:
# Sanity check
print(f"{len(crsp):,} stock-month rows, {crsp['month'].min()} through {crsp['month'].max()}")
crsp.head()

2,927,276 stock-month rows, 1995-01 through 2025-12


,permno,mthcaldt,mthprc,mthret,shrout,mthcap,is_eligible,month
0,10001,1995-01-31,7.75,-0.03125,2224,17236.0,True,1995-01
1,10002,1995-01-31,13.125,0.0,2999,39361.88,True,1995-01
2,10003,1995-01-31,2.125,-0.055556,5038,10705.75,True,1995-01
3,10009,1995-01-31,17.0,0.007407,1164,19788.0,True,1995-01
4,10010,1995-01-31,5.25,0.05,10359,54384.75,True,1995-01


Select and weight the top 500 each month

In [8]:
top500 = (
    crsp.loc[
        crsp["is_eligible"] & crsp["mthcap"].gt(0),
        ["month", "permno", "mthcap", "mthprc"],
    ]
    .sort_values(["month", "mthcap", "permno"], ascending=[True, False, True])
    .groupby("month", sort=False)
    .head(N_STOCKS)
    .copy()
)
top500["weight"] = (
    top500["mthcap"] / top500.groupby("month")["mthcap"].transform("sum")
)

monthly_index = (
    top500.groupby("month")
    .agg(n_stocks=("permno", "size"), total_market_cap=("mthcap", "sum"),
         weight_sum=("weight", "sum"))
)

print(f"Built {len(monthly_index)} monthly top-500 portfolios")

Built 372 monthly top-500 portfolios


Make sure the trailing 60 months prices are in the dataset

In [9]:
selected_permnos = top500["permno"].unique()
all_months = pd.period_range("1995-01", "2025-12", freq="M")
# Keep ALL months for stocks ever selected, including months after they leave the top 500.
# Do not merge these rows with month-specific top500 membership.
selected_history = crsp.loc[
    crsp["permno"].isin(selected_permnos),
    ["month", "permno", "mthprc", "mthret"],
].copy()

In [10]:
# Reindex the calendar so a missing month contributes zero to the rolling count.
price_grid = (
    selected_history.assign(has_price=lambda x: x["mthprc"].notna() & x["mthprc"].ne(0))
    .pivot(index="month", columns="permno", values="has_price")
    .reindex(index=all_months, columns=selected_permnos)
    .fillna(False)
)
return_grid = (
    selected_history.assign(has_return=lambda x: x["mthret"].notna())
    .pivot(index="month", columns="permno", values="has_return")
    .reindex(index=all_months, columns=selected_permnos)
    .fillna(False)
)

price_counts = price_grid.rolling(LOOKBACK, min_periods=1).sum()
return_counts = return_grid.rolling(LOOKBACK, min_periods=1).sum()
price_counts.index.name = return_counts.index.name = "month"
price_counts.columns.name = return_counts.columns.name = "permno"

coverage = (
    price_counts.stack().rename("n_price_months").reset_index()
    .merge(
        return_counts.stack().rename("n_return_months").reset_index(),
        on=["month", "permno"],
        validate="one_to_one",
    )
)
top500 = top500.merge(coverage, on=["month", "permno"], validate="one_to_one")
top500["has_60_prices"] = top500["n_price_months"].eq(LOOKBACK)
top500["has_60_returns"] = top500["n_return_months"].eq(LOOKBACK)

/var/folders/sb/cdw7gr054nvc469rwn15_4q00000gn/T/ipykernel_45148/1553463334.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [11]:
# Before December 1999, a full 60-month window does not exist in this download.
coverage_summary = (
    top500.loc[top500["month"] >= FIRST_FORMATION]
    .groupby("month")
    .agg(n_stocks=("permno", "size"),
         n_with_60_prices=("has_60_prices", "sum"),
         n_with_60_returns=("has_60_returns", "sum"))
)

print("Formation months:", len(coverage_summary))
print("Months with all 500 stocks having 60 prices:", coverage_summary["n_with_60_prices"].eq(500).sum())
coverage_summary.head()

Formation months: 313
Months with all 500 stocks having 60 prices: 0


,n_stocks,n_with_60_prices,n_with_60_returns
month,,,
1999-12,500,382,382
2000-01,500,379,379
2000-02,500,360,360
2000-03,500,369,369
2000-04,500,383,383


In [12]:
output_dir = Path("top500_data")
output_dir.mkdir(exist_ok=True)

formations_to_save = top500.copy()
formations_to_save["month"] = formations_to_save["month"].astype(str)
formations_to_save.to_csv(output_dir / "top500_formations.csv", index=False)

history_to_save = selected_history.copy()
history_to_save["month"] = history_to_save["month"].astype(str)
history_to_save.to_csv(output_dir / "selected_stock_monthly_history.csv", index=False)